# Rail Station Processing
This notebook cleans the raw LTA rail-station geometry file, assigns opening dates to MRT and LRT stations, attaches line metadata, and exports the processed rail dataset used in the resale joins.


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

df= geopandas.read_file("../data/raw/LTAMRTStationExitGEOJSON.geojson")
df.head()

# Drop unwanted rows (DT18 is Telok Ayer MRT and CC9 is Paya lebar MRT; both of which can already be found with their actual names in the dataset. 
# Marina South MRT has finished construction but is non-operational.
df = df[~df["STATION_NA"].isin(["DT18", "CC9", "MARINA SOUTH"])]

# Drop unwanted columns 
df = df.drop(columns=["OBJECTID", "INC_CRC", "FMEL_UPD_D"])

# Rename DT4 → HUME MRT STATION
df.loc[df["STATION_NA"] == "DT4", "STATION_NA"] = "HUME MRT STATION"

# Insert Date column BEFORE STATION_NA
df.insert(df.columns.get_loc("STATION_NA"), "Date", pd.NaT)

# Keeping only one entry for each station since each station has multiple exits
df = df.drop_duplicates(subset="STATION_NA")

# Drop exit code column since it is now redundant
df = df.drop(columns=["EXIT_CODE"])


## Split MRT and LRT Records
The source file contains both MRT and LRT exits. These cells separate the two groups and standardize the station names before opening dates are attached.


In [ ]:
# Split original dataframe into mrt and lrt
df_mrt = df[df["STATION_NA"].str.contains("MRT", case=False, na=False)].copy()
df_lrt = df[~df["STATION_NA"].str.contains("MRT", case=False, na=False)].copy()

# Clean station names
df_mrt["STATION_NA"] = df_mrt["STATION_NA"].str.replace(" MRT STATION", "", regex=False)
df_lrt["STATION_NA"] = df_lrt["STATION_NA"].str.replace(" LRT STATION", "", regex=False)

## Assign Opening Dates
The next blocks map stations to opening dates, using a common pre-2014 placeholder date for stations that were already operating before the resale sample begins.


In [ ]:
# Assign MRT opening dates (MRT stations that opened before 2014 are given an arbitrary opening date of 1/1/13 since their exact date does not matter)
mrt_dates = {
    "CANBERRA": "2019-11-02",
    "MARINA SOUTH PIER": "2014-11-23",
    "GUL CIRCLE": "2017-06-18",
    "TUAS CRESCENT": "2017-06-18",
    "TUAS WEST ROAD": "2017-06-18",
    "TUAS LINK": "2017-06-18",
    "PUNGGOL COAST": "2024-12-10",
    "BAYFRONT": "2012-01-14",
    "MARINA BAY": "2012-01-14",
    "BUGIS": "2013-12-22",
    "PROMENADE": "2013-12-22",
    "DOWNTOWN": "2013-12-22",
    "TELOK AYER": "2013-12-22",
    "CHINATOWN": "2013-12-22",
    "FORT CANNING": "2017-10-21",
    "BENCOOLEN": "2017-10-21",
    "JALAN BESAR": "2017-10-21",
    "BENDEEMER": "2017-10-21",
    "GEYLANG BAHRU": "2017-10-21",
    "MATTAR": "2017-10-21",
    "MACPHERSON": "2017-10-21",
    "UBI": "2017-10-21",
    "KAKI BUKIT": "2017-10-21",
    "BEDOK NORTH": "2017-10-21",
    "BEDOK RESERVOIR": "2017-10-21",
    "TAMPINES WEST": "2017-10-21",
    "TAMPINES": "2017-10-21",
    "TAMPINES EAST": "2017-10-21",
    "UPPER CHANGI": "2017-10-21",
    "WOODLANDS NORTH": "2020-01-31",
    "WOODLANDS SOUTH": "2020-01-31",
    "SPRINGLEAF": "2021-08-28",
    "LENTOR": "2021-08-28",
    "MAYFLOWER": "2021-08-28",
    "BRIGHT HILL": "2021-08-28",
    "UPPER THOMSON": "2021-08-28",
    "NAPIER": "2022-11-13",
    "ORCHARD BOULEVARD": "2022-11-13",
    "GREAT WORLD": "2022-11-13",
    "HAVELOCK": "2022-11-13",
    "MAXWELL": "2022-11-13",
    "SHENTON WAY": "2022-11-13",
    "GARDENS BY THE BAY": "2022-11-13",
    "TANJONG RHU": "2024-06-23",
    "KATONG PARK": "2024-06-23",
    "TANJONG KATONG": "2024-06-23",
    "MARINE PARADE": "2024-06-23",
    "MARINE TERRACE": "2024-06-23",
    "SIGLAP": "2024-06-23",
    "BAYSHORE": "2024-06-23",
    "BUKIT PANJANG": "2015-12-27",
    "CASHEW": "2015-12-27",
    "HILLVIEW": "2015-12-27",
    "BEAUTY WORLD": "2015-12-27",
    "KING ALBERT PARK": "2015-12-27",
    "SIXTH AVENUE": "2015-12-27",
    "TAN KAH KEE": "2015-12-27",
    "BOTANIC GARDENS": "2015-12-27",
    "STEVENS": "2015-12-27",
    "NEWTON": "2015-12-27",
    "LITTLE INDIA": "2015-12-27",
    "ROCHOR": "2015-12-27",
    "HUME": "2025-02-28"
}

df_mrt["Date"] = df_mrt["STATION_NA"].map(mrt_dates)
df_mrt["Date"] = pd.to_datetime(df_mrt["Date"]).fillna(pd.to_datetime("2013-01-01"))

# Insert Rail_type after STATION_NA
df_mrt.insert(df_mrt.columns.get_loc("STATION_NA") + 1, "Rail_type", "MRT")

In [ ]:
# Assign LRT opening dates (LRT stations that opened before 2014 are given an arbitrary opening date of 1/1/13 since their exact date does not matter)
lrt_dates = {
    "KUPANG": "2015-06-27",
    "NIBONG": "2014-06-29",
    "PUNGGOL POINT": "2016-02-29",
    "SAMUDERA": "2017-03-31",
    "SOO TECK": "2014-06-29",
    "SUMANG": "2014-06-29",
    "TECK LEE": "2024-08-15"
}

df_lrt["Date"] = df_lrt["STATION_NA"].str.upper().map(lrt_dates)
df_lrt["Date"] = pd.to_datetime(df_lrt["Date"]).fillna(pd.to_datetime("2013-01-01"))

df_lrt.insert(df_lrt.columns.get_loc("STATION_NA") + 1, "Rail_type", "LRT")

## Add Line Metadata and Geometry Formatting
After the two rail tables are recombined, the notebook harmonizes names, adds line labels, and converts the geometry into the projected CRS used elsewhere in the project.


In [ ]:
# Formatting
df_rail_final = pd.concat([df_mrt, df_lrt], ignore_index=True)
df_rail_final = df_rail_final.rename(columns={'STATION_NA': 'Station_name'})
df_rail_final['Station_name'] = df_rail_final['Station_name'].str.title()

# Complete station-to-line mapping
station_lines = {
    # East-West Line
    'Changi Airport': ['East West Line'],
    'Expo': ['East West Line', 'Downtown Line'],
    'Pasir Ris': ['East West Line'],
    'Tampines': ['East West Line', 'Downtown Line'],
    'Simei': ['East West Line'],
    'Tanah Merah': ['East West Line'],
    'Bedok': ['East West Line'],
    'Kembangan': ['East West Line'],
    'Eunos': ['East West Line'],
    'Paya Lebar': ['East West Line', 'Circle Line'],
    'Aljunied': ['East West Line'],
    'Kallang': ['East West Line'],
    'Lavender': ['East West Line'],
    'Bugis': ['East West Line', 'Downtown Line'],
    'City Hall': ['East West Line', 'North South Line'],
    'Raffles Place': ['East West Line', 'North South Line'],
    'Tanjong Pagar': ['East West Line'],
    'Outram Park': ['East West Line', 'North East Line', 'Thomson-East Coast Line'],
    'Tiong Bahru': ['East West Line'],
    'Redhill': ['East West Line'],
    'Queenstown': ['East West Line'],
    'Commonwealth': ['East West Line'],
    'Buona Vista': ['East West Line', 'Circle Line'],
    'Dover': ['East West Line'],
    'Clementi': ['East West Line'],
    'Jurong East': ['East West Line', 'North South Line'],
    'Chinese Garden': ['East West Line'],
    'Lakeside': ['East West Line'],
    'Boon Lay': ['East West Line'],
    'Pioneer': ['East West Line'],
    'Joo Koon': ['East West Line'],
    'Gul Circle': ['East West Line'],
    'Tuas Crescent': ['East West Line'],
    'Tuas West Road': ['East West Line'],
    'Tuas Link': ['East West Line'],

    # North-South Line
    'Bukit Batok': ['North South Line'],
    'Bukit Gombak': ['North South Line'],
    'Choa Chu Kang': ['North South Line', 'Bukit Panjang LRT'],
    'Yew Tee': ['North South Line'],
    'Kranji': ['North South Line'],
    'Marsiling': ['North South Line'],
    'Woodlands': ['North South Line', 'Thomson-East Coast Line'],
    'Admiralty': ['North South Line'],
    'Sembawang': ['North South Line'],
    'Canberra': ['North South Line'],
    'Yishun': ['North South Line'],
    'Khatib': ['North South Line'],
    'Yio Chu Kang': ['North South Line'],
    'Ang Mo Kio': ['North South Line'],
    'Bishan': ['North South Line', 'Circle Line'],
    'Braddell': ['North South Line'],
    'Toa Payoh': ['North South Line'],
    'Novena': ['North South Line'],
    'Newton': ['North South Line', 'Downtown Line'],
    'Orchard': ['North South Line', 'Thomson-East Coast Line'],
    'Somerset': ['North South Line'],
    'Dhoby Ghaut': ['North South Line', 'North East Line', 'Circle Line'],
    'Marina Bay': ['North South Line', 'Circle Line', 'Thomson-East Coast Line'],
    'Marina South Pier': ['North South Line'],

    # North East Line
    'HarbourFront': ['North East Line', 'Circle Line'],
    'Chinatown': ['North East Line', 'Downtown Line'],
    'Clarke Quay': ['North East Line'],
    'Little India': ['North East Line', 'Downtown Line'],
    'Farrer Park': ['North East Line'],
    'Boon Keng': ['North East Line'],
    'Potong Pasir': ['North East Line'],
    'Woodleigh': ['North East Line'],
    'Serangoon': ['North East Line', 'Circle Line'],
    'Kovan': ['North East Line'],
    'Hougang': ['North East Line'],
    'Buangkok': ['North East Line'],
    'Sengkang': ['North East Line', 'Sengkang LRT'],
    'Punggol': ['North East Line', 'Punggol LRT'],

    # Circle Line
    'Bras Basah': ['Circle Line'],
    'Esplanade': ['Circle Line'],
    'Promenade': ['Circle Line', 'Downtown Line'],
    'Nicoll Highway': ['Circle Line'],
    'Stadium': ['Circle Line'],
    'Mountbatten': ['Circle Line'],
    'Dakota': ['Circle Line'],
    'MacPherson': ['Circle Line', 'Downtown Line'],
    'Tai Seng': ['Circle Line'],
    'Bartley': ['Circle Line'],
    'Lorong Chuan': ['Circle Line'],
    'Marymount': ['Circle Line'],
    'Caldecott': ['Circle Line', 'Thomson-East Coast Line'],
    'Botanic Gardens': ['Circle Line', 'Downtown Line'],
    'Farrer Road': ['Circle Line'],
    'Holland Village': ['Circle Line'],
    'one-north': ['Circle Line'],
    'Kent Ridge': ['Circle Line'],
    'Haw Par Villa': ['Circle Line'],
    'Pasir Panjang': ['Circle Line'],
    'Labrador Park': ['Circle Line'],
    'Telok Blangah': ['Circle Line'],
    'Gardens by the Bay': ['Circle Line'],
    'Bayfront': ['Circle Line', 'Downtown Line'],

    # Downtown Line
    'Bukit Panjang': ['Downtown Line', 'Bukit Panjang LRT'],
    'Cashew': ['Downtown Line'],
    'Hillview': ['Downtown Line'],
    'Hume': ['Downtown Line'],
    'Beauty World': ['Downtown Line'],
    'King Albert Park': ['Downtown Line'],
    'Sixth Avenue': ['Downtown Line'],
    'Tan Kah Kee': ['Downtown Line'],
    'Stevens': ['Downtown Line', 'Thomson-East Coast Line'],
    'Rochor': ['Downtown Line'],
    'Downtown': ['Downtown Line'],
    'Telok Ayer': ['Downtown Line'],
    'Fort Canning': ['Downtown Line'],
    'Bencoolen': ['Downtown Line'],
    'Jalan Besar': ['Downtown Line'],
    'Bendemeer': ['Downtown Line'],
    'Geylang Bahru': ['Downtown Line'],
    'Mattar': ['Downtown Line'],
    'Ubi': ['Downtown Line'],
    'Kaki Bukit': ['Downtown Line'],
    'Bedok North': ['Downtown Line'],
    'Bedok Reservoir': ['Downtown Line'],
    'Tampines West': ['Downtown Line'],
    'Tampines East': ['Downtown Line'],
    'Upper Changi': ['Downtown Line'],

    # Thomson-East Coast Line
    'Woodlands North': ['Thomson-East Coast Line'],
    'Woodlands South': ['Thomson-East Coast Line'],
    'Springleaf': ['Thomson-East Coast Line'],
    'Lentor': ['Thomson-East Coast Line'],
    'Mayflower': ['Thomson-East Coast Line'],
    'Bright Hill': ['Thomson-East Coast Line'],
    'Upper Thomson': ['Thomson-East Coast Line'],
    'Napier': ['Thomson-East Coast Line'],
    'Orchard Boulevard': ['Thomson-East Coast Line'],
    'Great World': ['Thomson-East Coast Line'],
    'Havelock': ['Thomson-East Coast Line'],
    'Maxwell': ['Thomson-East Coast Line'],
    'Shenton Way': ['Thomson-East Coast Line'],
    'Tanjong Rhu': ['Thomson-East Coast Line'],
    'Katong Park': ['Thomson-East Coast Line'],
    'Tanjong Katong': ['Thomson-East Coast Line'],
    'Marine Parade': ['Thomson-East Coast Line'],
    'Marine Terrace': ['Thomson-East Coast Line'],
    'Siglap': ['Thomson-East Coast Line'],
    'Bayshore': ['Thomson-East Coast Line'],

    # Bukit Panjang LRT
    'Petir': ['Bukit Panjang LRT'],
    'Pending': ['Bukit Panjang LRT'],
    'Bangkit': ['Bukit Panjang LRT'],
    'Fajar': ['Bukit Panjang LRT'],
    'Segar': ['Bukit Panjang LRT'],
    'Jelapang': ['Bukit Panjang LRT'],
    'Senja': ['Bukit Panjang LRT'],
    'Ten Mile Junction': ['Bukit Panjang LRT'],
    'Phoenix': ['Bukit Panjang LRT'],
    'Keat Hong': ['Bukit Panjang LRT'],
    'Teck Whye': ['Bukit Panjang LRT'],
    'South View': ['Bukit Panjang LRT'],

    # Sengkang LRT
    'Compassvale': ['Sengkang LRT'],
    'Rumbia': ['Sengkang LRT'],
    'Bakau': ['Sengkang LRT'],
    'Kangkar': ['Sengkang LRT'],
    'Ranggung': ['Sengkang LRT'],
    'Cheng Lim': ['Sengkang LRT'],
    'Farmway': ['Sengkang LRT'],
    'Kupang': ['Sengkang LRT'],
    'Thanggam': ['Sengkang LRT'],
    'Fernvale': ['Sengkang LRT'],
    'Layar': ['Sengkang LRT'],
    'Tongkang': ['Sengkang LRT'],
    'Renjong': ['Sengkang LRT'],

    # Punggol LRT
    'Cove': ['Punggol LRT'],
    'Meridian': ['Punggol LRT'],
    'Coral Edge': ['Punggol LRT'],
    'Riviera': ['Punggol LRT'],
    'Kadaloor': ['Punggol LRT'],
    'Oasis': ['Punggol LRT'],
    'Damai': ['Punggol LRT'],
    'Sam Kee': ['Punggol LRT'],
    'Teck Lee': ['Punggol LRT'],
    'Punggol Point': ['Punggol LRT'],
    'Samudera': ['Punggol LRT'],
    'Nibong': ['Punggol LRT'],
    'Sumang': ['Punggol LRT'],
    'Soo Teck': ['Punggol LRT'],
}

# Normalise lookup
station_lines_norm = {k.strip().lower(): v for k, v in station_lines.items()}

def get_lines(station_name):
    if pd.isna(station_name):
        return []
    return station_lines_norm.get(str(station_name).strip().lower(), [])

# Expand rows — one row per line per station, filtered by Rail_type
expanded_rows = []
for _, row in df_rail_final.iterrows():
    lines = get_lines(row['Station_name'])
    rail_type = row['Rail_type']

    if lines:
        # Filter lines based on Rail_type
        if rail_type == 'LRT':
            filtered_lines = [line for line in lines if 'LRT' in line]
        else:  # MRT
            filtered_lines = [line for line in lines if 'LRT' not in line]

        # If after filtering there are matching lines, expand rows
        if filtered_lines:
            for line in filtered_lines:
                new_row = row.copy()
                new_row['Line_name'] = line
                expanded_rows.append(new_row)
        else:
            # No matching lines after filter, keep with null
            new_row = row.copy()
            new_row['Line_name'] = None
            expanded_rows.append(new_row)
    else:
        # Station not in mapping, keep with null
        new_row = row.copy()
        new_row['Line_name'] = None
        expanded_rows.append(new_row)

df_rail_final = pd.DataFrame(expanded_rows).reset_index(drop=True)

# Reorder columns to place Line_name just after Rail_type
cols = df_rail_final.columns.tolist()
cols.remove('Line_name')
rail_type_idx = cols.index('Rail_type')
cols.insert(rail_type_idx + 1, 'Line_name')
df_rail_final = df_rail_final[cols]

print(df_rail_final.head(5))

In [ ]:
# Convert mapping to 3414 
df_rail_final = gpd.GeoDataFrame(df_rail_final, geometry='geometry', crs='EPSG:4326')
df_rail_final = df_rail_final.to_crs("EPSG:3414")

df_rail_final.head()

## Inspect Missing Values and Export
The final checks flag any stations with incomplete metadata before the cleaned rail table is saved to data/processed/mrt_lrt_data.csv.


In [ ]:
df_rail_final[df_rail_final.isna().any(axis=1)]

In [ ]:
df_rail_final.to_csv("../data/processed/mrt_lrt_data.csv", index=False)